In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['AmazonLaptops'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [ ]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Vannessa"
datos_finales = []   # Se define fuera del try para que siempre exista
driver = None        # Se define fuera del try para poder cerrarlo con seguridad

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"  # Ruta del binario de Chrome

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://www.amazon.es/s?k=laptops")

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")

        # Espera fija para que la página termine de cargar
        time.sleep(10)

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[data-component-type='s-search-result']"
        )

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.TAG_NAME, "h2").text
                precio = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
            except:
                # Si el bloque no tiene nombre o precio, se salta
                continue

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CLASS_NAME, "s-pagination-next")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f" Extracción terminada: {len(datos_finales)} productos.")

except Exception as e:
    print(f" Error en Selenium: {e}")

finally:
    # Cierra el navegador solo si logró abrirse
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["AmazonLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpia el valor antes de convertirlo
            v_limpio = str(d["valor"]).replace(".", "").replace(",", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos cargados en MongoDB correctamente.")
    else:
        print(" No hay datos para guardar.")

except Exception as e:
    print(f" Error en MongoDB: {e}")
    

🧹 Limpieza de procesos y temporales completada.
 Navegador iniciado correctamente.
--- Procesando Página 1 ---
--- Procesando Página 2 ---
--- Procesando Página 3 ---


In [ ]:
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "G5_Turismo"
CATEGORIA_ACTUAL = "Hotel"
datos_finales = []
driver = None

# --- PASO 1: CONFIGURACION DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACION A BOOKING ---
    url = "https://www.booking.com/searchresults.es.html?ss=Santiago&checkin=2026-04-20&checkout=2026-04-21&group_adults=2"
    driver.get(url)

    # Pausa controlada para intervencion manual (VNC)
    print("\nAbre el entorno visual en: http://localhost:6080/vnc.html")
    print("Observa si aparece CAPTCHA, cookies o algo extraño.")
    input("Resuelve manualmente lo necesario y presiona ENTER para continuar...")

    # Cerrar pop-up de cookies si aparece
    try:
        cookie_btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        cookie_btn.click()
        print("Pop-up de cookies cerrado.")
        time.sleep(2)
    except:
        print("No se detecto pop-up de cookies.")

    # --- PASO 3: PAGINACION Y EXTRACCION ---
    limite_paginas = 3

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Pagina {nivel_pagina + 1} ---")
        time.sleep(5)

        # Esperar a que carguen los contenedores de hoteles
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "[data-testid='property-card']")
            )
        )

        bloques = driver.find_elements(By.CSS_SELECTOR, "[data-testid='property-card']")

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.CSS_SELECTOR, "[data-testid='title']").text.strip()
                precio_elem = bloque.find_element(By.CSS_SELECTOR, "[data-testid='price-and-discounted-price']")
                precio_texto = precio_elem.text.strip()

                datos_finales.append({
                    "item": nombre,
                    "valor_texto": precio_texto,
                    "categoria": CATEGORIA_ACTUAL,
                    "grupo": NOMBRE_GRUPO,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
                })
            except:
                continue

        print(f"Pagina {nivel_pagina + 1}: {len(bloques)} hoteles encontrados.")

        # Avanzar a la siguiente pagina
        try:
            btn_sig = driver.find_element(By.CSS_SELECTOR, "button[aria-label='Siguiente']")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontro el boton siguiente o es la ultima pagina.")
            break

    print(f"Extraccion terminada: {len(datos_finales)} hoteles en total.")

except Exception as e:
    print(f"Error en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 4: GUARDAR EN MONGODB ---
try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["proyecto_bigdata"]
    coleccion = db["datos_turismo"]

    if datos_finales:
        for d in datos_finales:
            # Limpiar precio y convertir a float
            import re
            numeros = re.findall(r"[\d\.]+", d["valor_texto"].replace(".", "").replace(",", "."))
            d["valor"] = float(numeros[0]) if numeros else 0.0
            del d["valor_texto"]  # Eliminar campo temporal

        coleccion.insert_many(datos_finales)
        print(f"Datos cargados en MongoDB correctamente. Total: {len(datos_finales)}")
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print(f"Error en MongoDB: {e}")

Limpieza de procesos y temporales completada.
Navegador iniciado correctamente.

Abre el entorno visual en: http://localhost:6080/vnc.html
Observa si aparece CAPTCHA, cookies o algo extraño.
